In [1]:
from dj_notebook import activate

plus = activate()

Output()

In [2]:
import pandas as pd
from django.db import transaction
from cuentas.models import User
from empresas.models import SucursalEmpresa, UsuarioEmpresa, Empresa


In [3]:
empresas = Empresa.objects.all()
for x in empresas:
    for y in SucursalEmpresa.objects.filter(empresa=x.pk):
        print(f'PK Empresa: {x.pk} | Nombre: {x.nombre} ||| Sucursales id: {y.pk} | {y.nombre}')


PK Empresa: 1 | Nombre: Snabbit Asesores Tecnologicos ||| Sucursales id: 1 | Oficina
PK Empresa: 2 | Nombre: Molina Rios Abogados ||| Sucursales id: 2 | Casa Matriz
PK Empresa: 3 | Nombre: Agricola Prodalmen ||| Sucursales id: 3 | Casa Matriz
PK Empresa: 4 | Nombre: Aldeas Infantiles SOS Chile ||| Sucursales id: 4 | Casa Matriz
PK Empresa: 5 | Nombre: Asipla Chile ||| Sucursales id: 5 | Oficina
PK Empresa: 6 | Nombre: Camara Española de Comercio ||| Sucursales id: 6 | Casa Matriz
PK Empresa: 7 | Nombre: AyG Asociados ||| Sucursales id: 7 | Oficina


In [4]:
# Cargar el Excel
df = pd.read_excel('usuarios_aygasociados.xlsx')

# Obtener la sucursal con pk=3 y definir el cargo por defecto
try:
    sucursal = SucursalEmpresa.objects.get(pk=7)
except SucursalEmpresa.DoesNotExist:
    raise Exception("La sucursal con pk=2 no existe.")

cargo_por_defecto = "Usuario Final"

# Recorrer cada fila del DataFrame
for index, row in df.iterrows():
    try:
        with transaction.atomic():
            # Extraer los datos necesarios del DataFrame
            email = row['Correo']
            first_name = row['Nombre']
            last_name = row['Apellido']
            rut = row.get('rut', None)
            # Puedes extraer más campos si es necesario

            # Crear el usuario
            user = User.objects.create_user(
                email=email,
                first_name=first_name,
                last_name=last_name,
                password='Hola.203040@111'
            )
            # Asignar campos opcionales
            if rut:
                user.rut = rut
            user.save()

            # Crear la vinculación en UsuarioEmpresa
            usuario_empresa = UsuarioEmpresa.objects.create(
                usuario=user,
                sucursal=sucursal,
                cargo=cargo_por_defecto
                # Puedes agregar fecha_ingreso, fecha_contrato o estado si lo requieres
            )

            print(f"Usuario {email} creado y vinculado a la sucursal {sucursal} con cargo '{cargo_por_defecto}'.")
    except Exception as e:
        print(f"Error creando usuario en la fila {index}: {e}")


Usuario naguileran@aygasociados.cl creado y vinculado a la sucursal Oficina de AyG Asociados con cargo 'Usuario Final'.
Usuario eaguileran@aygasociados.cl creado y vinculado a la sucursal Oficina de AyG Asociados con cargo 'Usuario Final'.
Usuario consultora@aygasociados.cl creado y vinculado a la sucursal Oficina de AyG Asociados con cargo 'Usuario Final'.
